# ORV fine-tune (Player + Ref) na Colab GPU
SCRUM-65/70/71. Trenira yolov8s na našem `dataset_pr` (Player+Ref).

**Pred zagonom:** Runtime → Change runtime type → **T4 GPU**.

**Naloži dataset:** datoteko `VID/dataset_pr.zip` (216 MB) prekopiraj v koren svojega **Google Drive** (MyDrive).

Nato samo zaženi vse celice (Runtime → Run all). Na koncu se prenese `best.pt`.

In [ ]:
!pip install -q ultralytics
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NI GPU — preklopi Runtime na T4!')

In [ ]:
# Dataset iz Google Drive (naloži dataset_pr.zip v koren MyDrive)
from google.colab import drive
drive.mount('/content/drive')
!cp '/content/drive/MyDrive/dataset_pr.zip' /content/
!unzip -q -o /content/dataset_pr.zip -d /content/
print('razpakirano:', __import__('os').listdir('/content/dataset_pr'))

In [ ]:
# data.yaml s Colab potmi (zamenja Windows poti iz zip-a)
yaml = '''train: /content/dataset_pr/train/images
val: /content/dataset_pr/valid/images
test: /content/dataset_pr/test/images

nc: 2
names: ['Player', 'Ref']
'''
open('/content/dataset_pr/data.yaml', 'w').write(yaml)
print(yaml)

In [ ]:
# Fine-tune (T4 zmore imgsz 960 + batch 16)
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data='/content/dataset_pr/data.yaml',
    epochs=80, imgsz=960, batch=16,
    patience=20, seed=42, name='orv_pr')

In [ ]:
# Vrednotenje na test množici (metrike za poročilo — SCRUM-71)
m = model.val(split='test')
print('mAP@0.5     :', round(float(m.box.map50), 4))
print('mAP@0.5:0.95:', round(float(m.box.map), 4))
for i, name in enumerate(['Player', 'Ref']):
    print(f'  {name:7} P={m.box.p[i]:.3f} R={m.box.r[i]:.3f} mAP50={m.box.ap50[i]:.3f}')

In [ ]:
# Prenesi naučene uteži (daj v VID/models/orv_pr.pt)
from google.colab import files
files.download('runs/detect/orv_pr/weights/best.pt')